# « Goutte » — film complet de 2 minutes (Blender, GPU)

1. *Settings* : **Accelerator GPU T4 ×2**, **Internet on** ; *Add-ons → Secrets* : cochez `GITHUB_TOKEN`.
2. **Save Version → Save & Run All (Commit)** : Kaggle calcule en arrière-plan (environ 6 à 9 h avec les réglages
   ci-dessous). Vous pouvez fermer la page.
3. Le film est dans l'onglet *Output* de la version : `films/goutte.mp4`.

In [ ]:
import os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

BRANCHE = 'claude/video-4k-high-fps-b7phx9'
CODE = '/kaggle/working/code'
try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    raise SystemExit("Secret GITHUB_TOKEN non attaché à CE notebook : menu Add-ons → Secrets, "
                     "cochez GITHUB_TOKEN, puis relancez cette cellule.")
shutil.rmtree(CODE, ignore_errors=True)   # toujours la dernière version du code
if True:
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCHE,
                        f'https://{token}@github.com/Amola07/Mon_Dashboard_Client.git', CODE],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***') or 'Code récupéré')
os.chdir(CODE)

In [ ]:
!pip install -q skia-python
!bash scripts/setup_tools.sh
os.environ['PATH'] = f"{CODE}/tools/ffmpeg/bin:" + os.environ['PATH']
!nvidia-smi --query-gpu=name,driver_version --format=csv

## Réglages
- `SCALE = 0.5` : rendu en 540×960 puis agrandi en 1080×1920 (tient dans une session de 12 h).
  `1.0` : plus net mais environ 24 h de calcul, trop long pour une seule session.
- `SAMPLES` : finesse de la lumière (96 = propre, débruité).

In [ ]:
SCALE = 0.5
SAMPLES = 96
# test : une image (la goutte et la pousse), pour vérifier le GPU et le temps par image
!python -m films.render_film --film goutte --scene film --scale {SCALE} --samples {SAMPLES} --still 1897 --out /kaggle/working/films

In [ ]:
!python -m films.render_film --film goutte --scene film --scale {SCALE} --samples {SAMPLES} --out /kaggle/working/films